In [4]:
# Step 1b: download GHCN-Daily for 100 Texas stations with complete TMAX
import pandas as pd, duckdb
st = pd.read_fwf('data/raw/ghcnd-stations.txt', colspecs=[(0,11),(12,20),(21,30),(38,68)], names=['STATION','LAT','LON','NAME'])
inv = pd.read_fwf('data/raw/ghcnd-inventory.txt', colspecs=[(0,11),(31,35),(36,40),(41,45)], names=['STATION','ELEMENT','FIRST','LAST'])
tx = st[st.STATION.str.startswith('USC00') | st.STATION.str.startswith('USW000')]
good = inv[(inv.ELEMENT == 'TMAX') & (inv.FIRST <= 1990) & (inv.LAST >= 2024)].STATION
picked = tx[tx.STATION.isin(good) & tx.LAT.between(25.8, 36.5) & tx.LON.between(-106.7, -93.5)].head(100)
con = duckdb.connect()
df = con.execute("""
    SELECT column0 AS STATION,
           column1 AS DATE,
           column2 AS ELEMENT,
           column3 AS DATA_VALUE,
           column5 AS Q_FLAG
    FROM read_csv('data/raw/*.csv.gz', header=false, null_padding=true)
""").df()

df = df[df.STATION.isin(picked.STATION)]
df['DATE'] = pd.to_datetime(df['DATE'].astype(str), format='%Y%m%d', errors='coerce')
print(df.shape, df.STATION.nunique())

(17883265, 5) 100


In [5]:
from ydata_profiling import ProfileReport
import pandas as pd
import os
# Tạo thư mục report nếu chưa có
os.makedirs('report', exist_ok=True)

# Lấy mẫu tối đa 20.000 dòng để profiling chạy mượt mà
sample = df.sample(min(20000, len(df)), random_state=42)
ProfileReport(sample, minimal=True).to_file('report/profile.html')
df.describe(include='all').T.to_csv('report/table_describe_raw.csv')

print('Đã tạo xong profile.html và table_describe_raw.csv thành công!')

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 216.98it/s]


Đã tạo xong profile.html và table_describe_raw.csv thành công!


In [6]:
# Step 2: pivot elements to columns, clean quality flags, convert units
df = df[df.Q_FLAG.isna()]                                   # drop values with a quality flag
w = df.pivot_table(index=['STATION', 'DATE'], columns='ELEMENT', values='DATA_VALUE').reset_index()
w[['TMAX', 'TMIN', 'PRCP']] = w[['TMAX', 'TMIN', 'PRCP']] / 10
w = w[(w.TMAX > -40) & (w.TMAX < 60)]
w = (w.set_index('DATE').groupby('STATION')[['TMAX','TMIN','PRCP']].resample('D').mean().reset_index())
w[['TMAX','TMIN']] = w.groupby('STATION')[['TMAX','TMIN']].transform(lambda x: x.interpolate(limit=3))
ok = w.groupby('STATION').TMAX.apply(lambda x: x.notna().mean()) >= 0.95
w = w[w.STATION.isin(ok[ok].index)]; w.to_parquet('data/processed/ghcn_tx.parquet', index=False)
print(w.shape, w.STATION.nunique())

(1325474, 5) 43


In [7]:
# Re-run the cleaning steps as functions so each one is logged (keep the same order as the cell above)
def clean_log(df0, steps):
    rows, d = [], df0.copy()
    for name, fn, why in steps:
        n0 = len(d); d = fn(d); rows.append([name, n0, len(d), n0 - len(d), why])
    return d, pd.DataFrame(rows, columns=['step', 'rows_before', 'rows_after', 'dropped', 'reason'])
_, cleaning_log = clean_log(w, [
    ('drop duplicates', lambda x: x.drop_duplicates(['STATION', 'DATE']), 'same unit and timestamp'),
    ('drop missing target', lambda x: x.dropna(subset=['TMAX']), 'cannot be forecast'),
])
cleaning_log.to_csv('report/table_cleaning_log.csv', index=False); print(cleaning_log)

                  step  rows_before  rows_after  dropped  \
0      drop duplicates      1325474     1325474        0   
1  drop missing target      1325474     1293612    31862   

                    reason  
0  same unit and timestamp  
1       cannot be forecast  


In [8]:
eda = w.copy()
import io
buf = io.StringIO(); eda.info(buf=buf); print(buf.getvalue()[:3000])
desc_num = eda.describe().T                         # numeric statistics
desc_num['skew'] = eda.select_dtypes('number').skew()  # skewness, > 1 means strongly right-skewed
desc_num.round(3).to_csv('report/table_describe_numeric.csv'); print(desc_num.round(3).head(15))
desc_cat = eda.select_dtypes(exclude='number').describe().T   # categorical columns: count, unique, top, freq
desc_cat.to_csv('report/table_describe_categorical.csv'); print(desc_cat.head(15))
notes = []                                          # record EDA notes for the paper
for c in eda.select_dtypes('number').columns:
    s = eda[c]
    if s.std() == 0: notes.append([c, 'constant column, drop'])
    if s.max() in (999, 9999, 99999, -9999): notes.append([c, 'agency missing code, replace with NaN'])
    if abs(s.skew()) > 2: notes.append([c, 'strongly skewed, consider log or tree models'])
pd.DataFrame(notes, columns=['column', 'note']).to_csv('report/table_eda_notes.csv', index=False); print(notes)

<class 'pandas.core.frame.DataFrame'>
Index: 1325474 entries, 48748 to 3410600
Data columns (total 5 columns):
 #   Column   Non-Null Count    Dtype         
---  ------   --------------    -----         
 0   STATION  1325474 non-null  object        
 1   DATE     1325474 non-null  datetime64[ns]
 2   TMAX     1293612 non-null  float64       
 3   TMIN     1291463 non-null  float64       
 4   PRCP     1254648 non-null  float64       
dtypes: datetime64[ns](1), float64(3), object(1)
memory usage: 60.7+ MB

             count                           mean                  min  \
ELEMENT                                                                  
DATE       1325474  1978-11-16 00:35:17.183739552  1891-01-01 00:00:00   
TMAX     1293612.0                      23.124117                -18.9   
TMIN     1291463.0                       7.852736                -36.7   
PRCP     1254648.0                       1.909884                  0.0   

                         25%              

In [ ]:
#EDA phần 2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

miss = eda.isna().mean().sort_values(ascending=False).rename('missing_rate').to_frame()
miss['decision'] = np.select([miss.missing_rate == 0, miss.missing_rate < 0.05, miss.missing_rate < 0.4],
                             ['keep', 'interpolate/impute median', 'drop or keep flag'], 'drop column')
miss.round(4).to_csv('report/table_missing.csv'); print(miss.head(15))
# missingness of the target by time and by unit
tcol, gcol = 'DATE', 'STATION'
by_time = eda.groupby(pd.to_datetime(eda[tcol], errors='coerce').dt.to_period('M') if not np.issubdtype(eda[tcol].dtype, np.number) else eda[tcol])['TMAX'].apply(lambda s: s.isna().mean())
by_unit = eda.groupby(gcol)['TMAX'].apply(lambda s: s.isna().mean()).sort_values(ascending=False)
print('target missing by period (top):', by_time.sort_values(ascending=False).head(5).round(3).to_dict())
print('target missing by unit (top):', by_unit.head(5).round(3).to_dict())
# co-missingness matrix: do columns go missing together
co = eda.isna().astype(int); co = co.loc[:, co.sum() > 0]
if co.shape[1] > 1: print(co.corr().round(2))
fig, ax = plt.subplots(figsize=(7, 3))
miss.missing_rate.head(12).plot.bar(ax=ax, color='#1B6B6D'); ax.set_ylabel('Missing rate'); ax.spines[['top','right']].set_visible(False)
fig.tight_layout(); fig.savefig('report/fig_eda_missing.png', dpi=300)

         missing_rate                   decision
ELEMENT                                         
PRCP         0.053434          drop or keep flag
TMIN         0.025659  interpolate/impute median
TMAX         0.024038  interpolate/impute median
STATION      0.000000                       keep
DATE         0.000000                       keep
target missing by period (top): {Period('1894-02', 'M'): 1.0, Period('1894-01', 'M'): 1.0, Period('1893-10', 'M'): 1.0, Period('1893-12', 'M'): 1.0, Period('1893-11', 'M'): 1.0}
target missing by unit (top): {'USC00340256': 0.049, 'USC00410493': 0.048, 'USC00037488': 0.045, 'USC00411048': 0.045, 'USC00411429': 0.043}
ELEMENT  TMAX  TMIN  PRCP
ELEMENT                  
TMAX     1.00  0.97  0.66
TMIN     0.97  1.00  0.65
PRCP     0.66  0.65  1.00


In [11]:
#EDA phần 3
num_cols = [c for c in ['TMAX', 'TMIN', 'PRCP'] if c in eda.columns]
corr = eda[num_cols].corr()
corr.round(3).to_csv('report/table_corr.csv'); print(corr.round(3))
cols = []
for c in eda.columns:
    if c == 'TMAX': cols.append([c, 'target', 'keep']); continue
    if c in ('DATE', 'STATION'): cols.append([c, 'time/unit key', 'keep for feature building']); continue
    if c in num_cols:
        r = corr.loc[c, 'TMAX'] if 'TMAX' in corr.columns else np.nan
        cols.append([c, f'correlation with target {r:.2f}', 'keep' if abs(r) > 0.05 or np.isnan(r) else 'consider dropping'])
    else: cols.append([c, 'other', 'review'])
pd.DataFrame(cols, columns=['column', 'reason', 'decision']).to_csv('report/table_columns.csv', index=False)
# target statistics by unit and by period
q = lambda s: pd.Series({'mean': s.mean(), 'median': s.median(), 'p05': s.quantile(0.05), 'p95': s.quantile(0.95), 'n': s.count()})
by_g = eda.groupby('STATION')['TMAX'].apply(q).unstack().sort_values('mean', ascending=False)
by_g.round(3).to_csv('report/table_target_by_unit.csv'); print(by_g.head(10).round(3))
period = eda['DATE'] if np.issubdtype(eda['DATE'].dtype, np.number) else pd.to_datetime(eda['DATE'], errors='coerce').dt.month
by_t = eda.groupby(period)['TMAX'].apply(q).unstack()
by_t.round(3).to_csv('report/table_target_by_period.csv'); print(by_t.round(3))

ELEMENT   TMAX  TMIN   PRCP
ELEMENT                    
TMAX     1.000  0.86 -0.017
TMIN     0.860  1.00  0.100
PRCP    -0.017  0.10  1.000
               mean  median   p05   p95        n
STATION                                         
USC00411720  28.401  30.000  13.3  38.9  15556.0
USC00410225  27.587  28.900  12.2  38.3  22552.0
USC00411048  26.362  27.800  10.6  37.2  45249.0
USC00410493  26.077  27.200   8.9  38.3  45127.0
USC00411875  26.069  27.200  10.0  38.3  45452.0
USC00411429  25.975  27.200  10.6  36.7  22825.0
USC00411800  25.835  26.833   8.9  38.3  41877.0
USC00299686  25.829  26.700  10.6  38.3  31070.0
USC00411596  25.728  27.200   9.4  37.2  31778.0
USC00410120  25.648  27.200   7.8  38.9  44344.0
        mean  median    p05   p95         n
DATE                                       
1     11.662    12.2  -0.60  22.8  109443.0
2     14.185    14.4   1.35  25.6  100261.0
3     18.632    19.4   6.10  28.9  110149.0
4     23.243    23.9  12.20  31.7  106502.0
5     27

In [13]:
# SQL statements run one by one in DuckDB; each SELECT prints its first 20 rows and is saved to report/table_q{k}.csv
import duckdb
if 'con' not in globals(): con = duckdb.connect('data/processed/ady.duckdb')   # reuse the connection opened in Step 1
SQL = r"""
-- Step 3: sql/queries.sql
CREATE OR REPLACE TABLE g AS SELECT * FROM 'data/processed/ghcn_tx.parquet';

-- Q1: per-station extreme threshold (95th percentile of summer TMAX, 1991-2020) and extreme days per decade
CREATE OR REPLACE TABLE thr AS
SELECT STATION, quantile_cont(TMAX, 0.95) AS thr FROM g
WHERE month(DATE) BETWEEN 6 AND 8 AND year(DATE) BETWEEN 1991 AND 2020 GROUP BY 1;

SELECT (yr / 10) * 10 AS decade, AVG(cnt) AS extreme_days_per_year FROM (
  SELECT g.STATION, year(g.DATE) AS yr, SUM(g.TMAX > thr.thr) AS cnt 
  FROM g JOIN thr ON g.STATION = thr.STATION GROUP BY g.STATION, year(g.DATE))
GROUP BY 1 ORDER BY 1;

-- Q2: fastest-warming stations (slope of extreme days per year)
SELECT STATION, regr_slope(cnt, yr) AS slope FROM (
  SELECT g.STATION, year(g.DATE) AS yr, SUM(g.TMAX > thr.thr) AS cnt 
  FROM g JOIN thr ON g.STATION = thr.STATION GROUP BY g.STATION, year(g.DATE))
GROUP BY 1 ORDER BY slope DESC LIMIT 10;

-- Q3: features and targets 1-3 days ahead (2020-2024 only for modelling)
CREATE OR REPLACE TABLE feat AS
SELECT g.STATION, g.DATE, g.TMAX, g.TMIN, g.PRCP, thr.thr, dayofyear(g.DATE) AS doy,
       LAG(g.TMAX, 1) OVER w AS tmax_lag1, LAG(g.TMAX, 2) OVER w AS tmax_lag2, LAG(g.TMAX, 7) OVER w AS tmax_lag7,
       LAG(g.TMIN, 1) OVER w AS tmin_lag1, AVG(g.TMAX) OVER (w ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS tmax_ma7,
       LEAD(g.TMAX, 1) OVER w AS y_1d, LEAD(g.TMAX, 2) OVER w AS y_2d, LEAD(g.TMAX, 3) OVER w AS y_3d
FROM g JOIN thr ON g.STATION = thr.STATION WHERE year(g.DATE) >= 2020 WINDOW w AS (PARTITION BY g.STATION ORDER BY g.DATE);
"""
k = 0
for stmt in SQL.split(';'):
    stmt = '\n'.join(l for l in stmt.splitlines() if not l.strip().startswith('--')).strip()
    if not stmt: continue
    res = con.execute(stmt)
    if stmt.upper().startswith(('SELECT', 'WITH')):
        k += 1; out = res.df(); out.to_csv(f'report/table_q{k}.csv', index=False); print(f'--- Q{k} ---'); print(out.head(20))
open('sql/queries.sql', 'w').write(SQL)

--- Q1 ---
    decade  extreme_days_per_year
0   1891.0               0.000000
1   1892.0               0.000000
2   1893.0               0.000000
3   1894.0               8.000000
4   1895.0               1.000000
5   1896.0              36.000000
6   1897.0               4.200000
7   1898.0               0.000000
8   1899.0               7.200000
9   1900.0               0.400000
10  1901.0               6.000000
11  1902.0               0.285714
12  1903.0               0.000000
13  1904.0               0.111111
14  1905.0               0.400000
15  1906.0               0.400000
16  1907.0               3.363636
17  1908.0               0.300000
18  1909.0               3.900000
19  1910.0               4.083333
--- Q2 ---
       STATION     slope
0  USC00298596  0.206651
1  USC00291931  0.146248
2  USC00298085  0.111328
3  USC00411429  0.108924
4  USC00411128  0.099519
5  USC00411430  0.097219
6  USC00297226  0.088999
7  USC00411033  0.085685
8  USC00410394  0.085052
9  USC00340567

1591

In [14]:
#3.2. Tiếp nối SQL
from scipy.stats import kruskal, spearmanr
q1 = con.execute("SELECT * FROM feat").df()          # feature table created in Q3
if np.issubdtype(q1['DATE'].dtype, np.number): q1['period'] = q1['DATE']            # numeric time key (year, cycle)
else: q1['period'] = pd.to_datetime(q1['DATE'], errors='coerce').dt.month             # datetime key: use month as period
# Table RQ1a: target by period
rq1a = q1.groupby('period')['TMAX'].agg(['mean', 'median', 'std', 'count']).round(3)
rq1a.to_csv('report/table_rq1a_period.csv'); print(rq1a)
groups = [s.values for _, s in q1.groupby('period')['TMAX'] if len(s) > 30]
if len(groups) > 1: print('Kruskal-Wallis across periods:', kruskal(*groups))
# Table RQ1b: Spearman correlation of the target with exogenous variables
ext = [c for c in q1.select_dtypes('number').columns if c not in ('TMAX', 'y_1d') and not c.startswith('TMAX')]
rows = []
for c in ext[:15]:
    r, pv = spearmanr(q1[c], q1['TMAX'], nan_policy='omit'); rows.append([c, round(r, 3), pv])
rq1b = pd.DataFrame(rows, columns=['variable', 'spearman_r', 'p_value']).sort_values('spearman_r', key=abs, ascending=False)
rq1b.to_csv('report/table_rq1b_corr.csv', index=False); print(rq1b)
# Table RQ1c: target by unit, top and bottom 5
rq1c = q1.groupby('STATION')['TMAX'].agg(['mean', 'count']).sort_values('mean', ascending=False)
pd.concat([rq1c.head(5), rq1c.tail(5)]).round(3).to_csv('report/table_rq1c_units.csv')
print('RQ1 CONCLUSION (fill with real numbers): highest period', rq1a['mean'].idxmax(), 'is', round(rq1a['mean'].max() / max(rq1a['mean'].min(), 1e-9), 2), 'times the lowest period')

          mean  median    std  count
period                              
1       11.460    11.7  6.915   8802
2       14.224    15.0  8.504   8149
3       19.940    20.6  6.823   8864
4       23.302    23.9  5.981   8621
5       27.447    27.8  5.321   8739
6       32.060    32.8  4.556   8670
7       33.719    33.9  4.020   8781
8       33.933    34.4  4.420   8555
9       30.562    31.1  4.899   7821
10      25.123    26.1  6.501   7476
11      18.427    18.9  6.562   7214
12      14.885    15.0  6.562   7346
Kruskal-Wallis across periods: KruskalResult(statistic=np.float64(nan), pvalue=np.float64(nan))
     variable  spearman_r       p_value
8    tmax_ma7       0.914  0.000000e+00
4   tmax_lag1       0.911  0.000000e+00
0        TMIN       0.865  0.000000e+00
9        y_2d       0.850  0.000000e+00
5   tmax_lag2       0.850  0.000000e+00
7   tmin_lag1       0.836  0.000000e+00
10       y_3d       0.818  0.000000e+00
6   tmax_lag7       0.784  0.000000e+00
11     period       0.253 

In [15]:
#Bước 5.phần A
import seaborn as sns
def style(ax): ax.spines[['top', 'right']].set_visible(False)
d = q1.dropna(subset=['TMAX'])
# (1) distribution
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].hist(d['TMAX'], bins=50, color='#1B6B6D'); ax[0].set_xlabel('TMAX °C'); ax[0].set_ylabel('Count'); style(ax[0])
ax[1].boxplot(d['TMAX'], vert=False); ax[1].set_xlabel('TMAX'); style(ax[1])
fig.tight_layout(); fig.savefig('report/fig_rq1_distribution.png', dpi=300); plt.close(fig)
# (2) time series of the first 3 units
units = d['STATION'].unique()[:3]
fig, ax = plt.subplots(figsize=(9, 3))
for u in units:
    s = d[d['STATION'] == u].sort_values('DATE'); ax.plot(s['DATE'], s['TMAX'], lw=0.8, label=str(u))
ax.set_ylabel('TMAX'); ax.legend(frameon=False); style(ax); fig.tight_layout(); fig.savefig('report/fig_rq1_timeseries.png', dpi=300); plt.close(fig)
# (3) heatmap period x unit (top 15 units)
top = d.groupby('STATION')['TMAX'].mean().nlargest(15).index
hm = d[d['STATION'].isin(top)].pivot_table(index='STATION', columns='period', values='TMAX', aggfunc='mean')
fig, ax = plt.subplots(figsize=(9, 4)); sns.heatmap(hm, cmap='YlGnBu', ax=ax, cbar_kws={'label': 'TMAX'}); ax.set_xlabel('Period'); ax.set_ylabel('Unit')
fig.tight_layout(); fig.savefig('report/fig_rq1_heatmap.png', dpi=300); plt.close(fig)
# (4) boxplot by period
fig, ax = plt.subplots(figsize=(9, 3)); sns.boxplot(data=d, x='period', y='TMAX', color='#9FBFBF', ax=ax, showfliers=False); style(ax)
fig.tight_layout(); fig.savefig('report/fig_rq1_box_period.png', dpi=300); plt.close(fig)
# (5) scatter against the strongest exogenous variable
if len(rq1b):
    v = rq1b.iloc[0].variable
    fig, ax = plt.subplots(figsize=(5, 3.5)); ax.scatter(d[v], d['TMAX'], s=4, alpha=0.3, color='#1B6B6D'); ax.set_xlabel(v); ax.set_ylabel('TMAX'); style(ax)
    fig.tight_layout(); fig.savefig('report/fig_rq1_scatter.png', dpi=300); plt.close(fig)
# (6) correlation heatmap
fig, ax = plt.subplots(figsize=(6, 5)); sns.heatmap(d.select_dtypes('number').corr().round(2), cmap='coolwarm', center=0, ax=ax, annot=False)
fig.tight_layout(); fig.savefig('report/fig_rq1_corr.png', dpi=300); plt.close(fig)
# (7) group comparison with labelled bars
gm = d.groupby('period')['TMAX'].mean()
fig, ax = plt.subplots(figsize=(8, 3)); bars = ax.bar(gm.index.astype(str), gm.values, color='#1B6B6D'); ax.bar_label(bars, fmt='%.1f', fontweight='bold', fontsize=8); ax.set_ylabel('Mean TMAX'); style(ax)
fig.tight_layout(); fig.savefig('report/fig_rq1_period_bar.png', dpi=300); plt.close(fig)
print('7 figures saved to report/. Write each caption as: conclusion + number + condition.')

7 figures saved to report/. Write each caption as: conclusion + number + condition.


In [16]:
#Test case của Notebook_A
import hashlib, json, os
feat = con.execute("SELECT * FROM feat").df()
assert len(feat) >= 30000, 'fewer than 30k rows, widen the scope'
assert feat['y_1d'].notna().sum() > 0.8 * len(feat), 'too many missing targets'
key = ['STATION', 'DATE']
assert not feat.duplicated(key).any(), 'duplicate unit-time keys'
lead_cols = [c for c in feat.columns if c.startswith('y_') and c != 'y_1d']
assert all(c.startswith('y_') for c in lead_cols), 'only target columns may contain future values'
print('Tests A: OK')
feat.to_parquet('data/processed/feat.parquet', index=False)
h = hashlib.md5(open('data/processed/feat.parquet', 'rb').read()).hexdigest()
manifest = {'rows': int(len(feat)), 'cols': list(feat.columns), 'md5': h,
            'time_min': str(feat['DATE'].min()), 'time_max': str(feat['DATE'].max()), 'author': 'Student A'}
json.dump(manifest, open('data/processed/manifest.json', 'w'), indent=2, default=str); print(manifest)

Tests A: OK
{'rows': 103602, 'cols': ['STATION', 'DATE', 'TMAX', 'TMIN', 'PRCP', 'thr', 'doy', 'tmax_lag1', 'tmax_lag2', 'tmax_lag7', 'tmin_lag1', 'tmax_ma7', 'y_1d', 'y_2d', 'y_3d'], 'md5': '7a8243d8fb68bd94d73ae4d8dc2f3d79', 'time_min': '2020-01-01 00:00:00', 'time_max': '2026-09-20 00:00:00', 'author': 'Student A'}
